# 07 · Cache & Storage Levels in a Real Cluster (Case B)

**Theory**: docs/06-persistence-and-optimization.md

**Prerequisite**: `make up-cluster` still running.

In [ ]:
import sys
import time

sys.path.insert(0, "../scripts")
from lab_utils import get_connect_session, layer_path
from pyspark import StorageLevel

spark = get_connect_session("07-cache-storage-levels")
vendas = spark.read.parquet(layer_path("connect", "bronze", "vendas"))

## `MEMORY_ONLY` vs. `MEMORY_AND_DISK`

`persist()` lets you pick exactly where cached data lives. Check the Spark
UI's **Storage** tab (http://localhost:4040/storage/) after each cell to see
the DataFrame actually materialized there, with its size and storage
fraction.

In [ ]:
vendas.persist(StorageLevel.MEMORY_ONLY)
vendas.count()  # materializes the cache
print("Cached with MEMORY_ONLY — check the Storage tab now.")

In [ ]:
vendas.unpersist()

vendas.persist(StorageLevel.MEMORY_AND_DISK)
vendas.count()
print("Cached with MEMORY_AND_DISK — check the Storage tab again.")
vendas.unpersist()

## Adaptive Query Execution (AQE) and data skew

We build a deliberately skewed slice — 90% of rows land in a single region —
then aggregate with AQE on and off. Watch the Spark UI Stages tab: with AQE
on, Spark splits the skewed partition into smaller sub-tasks (Skew Join
Optimization) instead of leaving one Task to do 90% of the work alone.

In [ ]:
from pyspark.sql.functions import rand, when

skewed = vendas.withColumn(
    "regiao_skewed",
    when(rand(seed=42) < 0.9, "Sudeste").otherwise(vendas.regiao),
)
skewed.createOrReplaceTempView("vendas_skewed")

for aqe in (False, True):
    spark.conf.set("spark.sql.adaptive.enabled", str(aqe).lower())
    start = time.perf_counter()
    spark.sql(
        "SELECT regiao_skewed, SUM(valor) FROM vendas_skewed GROUP BY regiao_skewed"
    ).collect()
    elapsed = time.perf_counter() - start
    print(f"AQE={aqe!s:5s} -> {elapsed:.2f}s (see Spark UI Stages tab for task-level detail)")

spark.conf.set("spark.sql.adaptive.enabled", "true")  # restore default

In [ ]:
spark.stop()